# Modeling

Feature Engineering 단계에서 생성한 시간 기준 Train/Test 데이터를 사용하여
서울 아파트 전월세 가격 예측 모델을 구축한다.

## 모델링 목표

### 전세 모델
- Target: 전세보증금
- Train: 2023~2024
- Test: 2025

### 월세 모델
- Target: 월세
- Train: 2023~2024
- Test: 2025

## 모델 비교 전략

복잡한 모델의 성능이 실제로 얼마나 개선되는지 확인하기 위해
단순한 Baseline 모델부터 단계적으로 비교한다.

1. Baseline
2. Linear Model
3. Tree-based Model
4. XGBoost

## 평가 지표

- MAE: 평균 절대 오차
- RMSE: 큰 오차에 더 큰 패널티를 주는 지표
- R²: 실제 가격 변동을 모델이 설명하는 정도

모델은 로그 변환된 Target으로 학습하되,
최종 평가는 원래 가격 단위로 복원하여 수행한다.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

MODEL_DATA_DIR = Path("../data/processed/modeling")

jeonse_train = pd.read_csv(
    MODEL_DATA_DIR / "jeonse_train.csv",
    parse_dates=["contract_date"]
)

jeonse_test = pd.read_csv(
    MODEL_DATA_DIR / "jeonse_test.csv",
    parse_dates=["contract_date"]
)

monthly_train = pd.read_csv(
    MODEL_DATA_DIR / "monthly_train.csv",
    parse_dates=["contract_date"]
)

monthly_test = pd.read_csv(
    MODEL_DATA_DIR / "monthly_test.csv",
    parse_dates=["contract_date"]
)

print("전세 Train:", jeonse_train.shape)
print("전세 Test :", jeonse_test.shape)

print()
print("월세 Train:", monthly_train.shape)
print("월세 Test :", monthly_test.shape)

전세 Train: (266848, 11)
전세 Test : (238570, 11)

월세 Train: (185103, 12)
월세 Test : (183778, 12)


In [2]:
def evaluate_regression(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_true, y_pred)

    result = {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

    print(f"[{model_name}]")
    print(f"MAE : {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"R²  : {r2:.4f}")

    return result

## 전세 Baseline

머신러닝 모델이 실제로 의미 있는 성능 향상을 제공하는지 확인하기 위해
Train 데이터의 전세보증금 중앙값을 모든 Test 데이터에 예측하는
Median Baseline을 설정한다.

향후 모든 모델은 이 Baseline보다 낮은 MAE와 RMSE,
높은 R²를 보여야 의미 있는 개선으로 판단한다.

In [3]:
jeonse_median = jeonse_train["보증금(만원)"].median()

baseline_pred = np.full(
    len(jeonse_test),
    jeonse_median
)

baseline_result = evaluate_regression(
    jeonse_test["보증금(만원)"],
    baseline_pred,
    "Median Baseline"
)

print()
print(
    "Train 전세보증금 중앙값:",
    f"{jeonse_median:,.0f}만원"
)

[Median Baseline]
MAE : 24,418.73
RMSE: 38,372.88
R²  : -0.0898

Train 전세보증금 중앙값: 48,620만원


## 전세 Ridge Regression

첫 번째 학습 모델로 Ridge Regression을 사용한다.

범주형 변수인 자치구와 법정동은 One-Hot Encoding을 적용하고,
수치형 변수는 StandardScaler를 이용해 표준화한다.

전세보증금은 오른쪽으로 치우친 분포를 가지므로
`log_deposit`을 Target으로 학습한다.

최종 성능 평가는 예측값을 원래 보증금 단위로 복원한 뒤
MAE, RMSE, R²를 계산한다.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.linear_model import Ridge
from sklearn import set_config

set_config(display="text")

In [5]:
categorical_features = [
    "자치구명",
    "법정동명"
]

numeric_features = [
    "임대면적",
    "building_age",
    "층",
    "time_idx",
    "contract_month_sin",
    "contract_month_cos"
]

features = (
    categorical_features
    + numeric_features
)

In [6]:
X_train = jeonse_train[features]
X_test = jeonse_test[features]

y_train_log = jeonse_train["log_deposit"]

y_test = jeonse_test["보증금(만원)"]

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [8]:
ridge_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            Ridge(alpha=1.0)
        )
    ]
)

In [11]:
ridge_model.fit(
    X_train,
    y_train_log
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['자치구명', '법정동명']),
                                                 ('num', StandardScaler(),
                                                  ['임대면적', 'building_age', '층',
                                                   'time_idx',
                                                   'contract_month_sin',
                                                   'contract_month_cos'])])),
                ('model', Ridge())])

In [12]:
ridge_pred_log = ridge_model.predict(
    X_test
)

ridge_pred = np.expm1(
    ridge_pred_log
)

In [13]:
ridge_result = evaluate_regression(
    y_test,
    ridge_pred,
    "Ridge Regression"
)

[Ridge Regression]
MAE : 11,750.89
RMSE: 21,799.32
R²  : 0.6483


In [14]:
mae_improvement = (
    baseline_result["MAE"]
    - ridge_result["MAE"]
) / baseline_result["MAE"] * 100

print(
    "Baseline 대비 MAE 개선율:",
    f"{mae_improvement:.2f}%"
)

Baseline 대비 MAE 개선율: 51.88%


### Ridge Regression 결과

Ridge Regression의 전세가격 예측 성능은 다음과 같다.

- MAE: 11,750.89만원
- RMSE: 21,799.32만원
- R²: 0.6483
- Median Baseline 대비 MAE 개선율: 51.88%

단순히 Train 데이터의 중앙값을 사용하는 Baseline과 비교했을 때
MAE가 약 51.9% 감소하였다.

또한 R²는 0.6483으로,
테스트 데이터의 전세가격 변동 중 약 64.8%를
선형 모델이 설명하는 것으로 나타났다.

이는 지역, 면적, 건축연식, 층, 시간 정보만으로도
전세가격의 상당 부분을 설명할 수 있음을 보여준다.

다만 부동산 가격에는 변수 간 비선형 관계와 상호작용이 존재할 가능성이 있으므로
다음 단계에서는 Tree 기반 모델을 사용하여 성능 향상 여부를 확인한다.

## 전세 Random Forest

선형 회귀에서는 각 Feature와 가격 사이의 관계를
주로 선형적인 형태로 모델링한다.

Random Forest는 여러 개의 결정트리를 결합하여
면적, 지역, 건축연식 등 변수 사이의 비선형 관계와
상호작용을 학습할 수 있다.

Ridge Regression과 동일한 Train/Test 데이터를 사용하여
Tree 기반 모델의 성능 향상 여부를 비교한다.

In [15]:
from sklearn.ensemble import RandomForestRegressor

In [16]:
rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=5,
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

In [17]:
rf_model.fit(
    X_train,
    y_train_log
);

In [18]:
rf_pred_log = rf_model.predict(
    X_test
)

rf_pred = np.expm1(
    rf_pred_log
)

In [19]:
rf_result = evaluate_regression(
    y_test,
    rf_pred,
    "Random Forest"
)

[Random Forest]
MAE : 10,085.60
RMSE: 16,266.84
R²  : 0.8042


In [20]:
rf_vs_ridge = (
    ridge_result["MAE"]
    - rf_result["MAE"]
) / ridge_result["MAE"] * 100

rf_vs_baseline = (
    baseline_result["MAE"]
    - rf_result["MAE"]
) / baseline_result["MAE"] * 100

print(
    "Ridge 대비 MAE 개선율:",
    f"{rf_vs_ridge:.2f}%"
)

print(
    "Baseline 대비 MAE 개선율:",
    f"{rf_vs_baseline:.2f}%"
)

Ridge 대비 MAE 개선율: 14.17%
Baseline 대비 MAE 개선율: 58.70%


### Random Forest 결과

Random Forest의 전세가격 예측 성능은 다음과 같다.

- MAE: 10,085.60만원
- RMSE: 16,266.84만원
- R²: 0.8042
- Ridge 대비 MAE 개선율: 14.17%
- Median Baseline 대비 MAE 개선율: 58.70%

Random Forest는 Ridge Regression보다
MAE와 RMSE 모두 낮은 성능을 보였으며,
R² 역시 0.6483에서 0.8042로 크게 향상되었다.

이는 전세가격과 지역, 면적, 건축연식 등의 관계가
단순한 선형 관계만으로 설명되지 않으며,
변수 간 비선형 관계와 상호작용이
가격 예측에 중요한 역할을 한다는 것을 보여준다.

특히 RMSE가 크게 감소한 점을 고려하면
고가 전세 계약에서 발생하는 큰 예측 오차 역시
Ridge Regression보다 줄어든 것으로 판단할 수 있다.

다음 단계에서는 Gradient Boosting 기반의 XGBoost를 적용하여
추가적인 성능 향상이 가능한지 확인한다.

## 전세 XGBoost

XGBoost는 이전 트리의 예측 오차를 다음 트리가 보완하는
Gradient Boosting 방식의 모델이다.

Random Forest와 마찬가지로 비선형 관계를 학습할 수 있으며,
각 트리가 이전 모델의 잔차를 순차적으로 개선한다는 차이가 있다.

동일한 Train/Test 데이터와 Feature를 사용하여
Random Forest 대비 추가적인 성능 향상이 있는지 확인한다.

In [21]:
from xgboost import XGBRegressor

In [22]:
xgb_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBRegressor(
                n_estimators=400,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                tree_method="hist",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

In [23]:
xgb_model.fit(
    X_train,
    y_train_log
);

In [24]:
xgb_pred_log = xgb_model.predict(
    X_test
)

xgb_pred = np.expm1(
    xgb_pred_log
)

In [25]:
xgb_result = evaluate_regression(
    y_test,
    xgb_pred,
    "XGBoost"
)

[XGBoost]
MAE : 9,074.27
RMSE: 14,690.37
R²  : 0.8403


In [26]:
xgb_vs_rf = (
    rf_result["MAE"]
    - xgb_result["MAE"]
) / rf_result["MAE"] * 100

xgb_vs_baseline = (
    baseline_result["MAE"]
    - xgb_result["MAE"]
) / baseline_result["MAE"] * 100

print(
    "Random Forest 대비 MAE 개선율:",
    f"{xgb_vs_rf:.2f}%"
)

print(
    "Baseline 대비 MAE 개선율:",
    f"{xgb_vs_baseline:.2f}%"
)

Random Forest 대비 MAE 개선율: 10.03%
Baseline 대비 MAE 개선율: 62.84%


### XGBoost 결과

XGBoost의 전세가격 예측 성능은 다음과 같다.

- MAE: 9,074.27만원
- RMSE: 14,690.37만원
- R²: 0.8403
- Random Forest 대비 MAE 개선율: 10.03%
- Median Baseline 대비 MAE 개선율: 62.84%

현재 비교한 모델 가운데 XGBoost가 가장 낮은 MAE와 RMSE,
가장 높은 R²를 기록하였다.

특히 Median Baseline과 비교하면 MAE가 약 62.8% 감소하였으며,
Random Forest와 비교해도 약 10.0% 추가적인 개선이 나타났다.

R²는 0.8403으로,
2025년 Test 데이터의 전세가격 변동 중 약 84.0%를
현재 모델이 설명하는 것으로 나타났다.

따라서 현재 설정한 Feature와 모델 가운데
XGBoost를 전세가격 예측의 최종 후보 모델로 선정한다.

다만 본 결과는 현재 설정한 하이퍼파라미터 기준이며,
향후 교차검증 및 하이퍼파라미터 튜닝을 통해
추가적인 성능 개선 가능성을 검토할 수 있다.

In [27]:
jeonse_model_results = pd.DataFrame([
    baseline_result,
    ridge_result,
    rf_result,
    xgb_result
])

jeonse_model_results = (
    jeonse_model_results
    .sort_values("MAE")
    .reset_index(drop=True)
)

jeonse_model_results

,Model,MAE,RMSE,R2
0,XGBoost,9074.272461,14690.369635,0.840278
1,Random Forest,10085.595599,16266.836257,0.804158
2,Ridge Regression,11750.888721,21799.320249,0.648289
3,Median Baseline,24418.729723,38372.875753,-0.089806


## 전세 모델 최종 비교

네 가지 모델을 동일한 2025년 Test 데이터에서 비교한 결과,
XGBoost가 모든 주요 평가 지표에서 가장 높은 성능을 보였다.

Tree 기반 모델인 Random Forest와 XGBoost가
선형 모델인 Ridge보다 높은 성능을 보인 점을 통해,
서울 아파트 전세가격에는 변수 간 비선형 관계와
상호작용이 중요한 역할을 하는 것으로 판단된다.

현재 전세가격 예측의 최종 모델로 XGBoost를 선정한다.

# 월세 예측 모델

전세 모델과 동일하게 2023~2024년 데이터를 학습에 사용하고,
2025년 계약 데이터를 Test 데이터로 사용한다.

월세 모델에서는 보증금이 월세 결정과 밀접하게 관련되므로
로그 변환된 보증금을 Feature에 포함한다.

Target은 로그 변환된 월세(`log_rent`)를 사용하고,
최종 평가에서는 다시 원래 월세 단위로 복원한다.

In [28]:
monthly_categorical_features = [
    "자치구명",
    "법정동명"
]

monthly_numeric_features = [
    "임대면적",
    "building_age",
    "층",
    "log_deposit",
    "time_idx",
    "contract_month_sin",
    "contract_month_cos"
]

monthly_features = (
    monthly_categorical_features
    + monthly_numeric_features
)

X_monthly_train = monthly_train[monthly_features]
X_monthly_test = monthly_test[monthly_features]

y_monthly_train_log = monthly_train["log_rent"]
y_monthly_test = monthly_test["임대료(만원)"]

In [29]:
monthly_median = monthly_train[
    "임대료(만원)"
].median()

monthly_baseline_pred = np.full(
    len(monthly_test),
    monthly_median
)

monthly_baseline_result = evaluate_regression(
    y_monthly_test,
    monthly_baseline_pred,
    "Monthly Median Baseline"
)

print()
print(
    "Train 월세 중앙값:",
    f"{monthly_median:,.0f}만원"
)

[Monthly Median Baseline]
MAE : 75.15
RMSE: 131.83
R²  : -0.1049

Train 월세 중앙값: 72만원


### 월세 Baseline 결과

Train 데이터의 월세 중앙값인 72만원을
모든 Test 데이터에 동일하게 예측한 결과:

- MAE: 75.15만원
- RMSE: 131.83만원
- R²: -0.1049

단순 중앙값 예측은 2025년 월세 가격 변동을
설명하지 못하는 것으로 나타났다.

따라서 이후 학습 모델은
Baseline보다 낮은 MAE와 RMSE,
높은 R²를 보이는지를 기준으로 평가한다.

In [30]:
monthly_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            monthly_categorical_features
        ),
        (
            "num",
            StandardScaler(),
            monthly_numeric_features
        )
    ]
)

In [31]:
monthly_ridge_model = Pipeline(
    steps=[
        (
            "preprocessor",
            monthly_preprocessor
        ),
        (
            "model",
            Ridge(alpha=1.0)
        )
    ]
)

In [32]:
monthly_ridge_model.fit(
    X_monthly_train,
    y_monthly_train_log
);

In [33]:
monthly_ridge_pred_log = monthly_ridge_model.predict(
    X_monthly_test
)

monthly_ridge_pred = np.expm1(
    monthly_ridge_pred_log
)

In [34]:
monthly_ridge_result = evaluate_regression(
    y_monthly_test,
    monthly_ridge_pred,
    "Monthly Ridge Regression"
)

[Monthly Ridge Regression]
MAE : 57.81
RMSE: 169.70
R²  : -0.8308


In [35]:
monthly_ridge_improvement = (
    monthly_baseline_result["MAE"]
    - monthly_ridge_result["MAE"]
) / monthly_baseline_result["MAE"] * 100

print(
    "Baseline 대비 MAE 개선율:",
    f"{monthly_ridge_improvement:.2f}%"
)

Baseline 대비 MAE 개선율: 23.07%


In [36]:
monthly_diagnostic = monthly_test[
    [
        "임대료(만원)",
        "자치구명",
        "법정동명",
        "임대면적",
        "building_age",
        "log_deposit"
    ]
].copy()

monthly_diagnostic["prediction"] = monthly_ridge_pred

monthly_diagnostic["absolute_error"] = np.abs(
    monthly_diagnostic["임대료(만원)"]
    - monthly_diagnostic["prediction"]
)

In [37]:
quantiles = [
    0.50,
    0.90,
    0.95,
    0.99,
    0.995,
    0.999
]

print("실제 월세:")
print(
    monthly_diagnostic["임대료(만원)"]
    .quantile(quantiles)
)

print()
print("예측 월세:")
print(
    monthly_diagnostic["prediction"]
    .quantile(quantiles)
)

print()
print(
    "실제 최대:",
    monthly_diagnostic["임대료(만원)"].max()
)

print(
    "예측 최대:",
    monthly_diagnostic["prediction"].max()
)

실제 월세:
0.500      77.0
0.900     250.0
0.950     325.0
0.990     560.0
0.995     700.0
0.999    1200.0
Name: 임대료(만원), dtype: float64

예측 월세:
0.500      66.349898
0.900     182.606367
0.950     256.781232
0.990     585.346297
0.995     893.533642
0.999    2725.086528
Name: prediction, dtype: float64

실제 최대: 4000
예측 최대: 16047.97094843168


In [38]:
print(
    monthly_diagnostic["absolute_error"]
    .quantile([
        0.50,
        0.90,
        0.95,
        0.99,
        0.999
    ])
)

0.500      29.229494
0.900     125.343564
0.950     176.985228
0.990     371.270192
0.999    1948.799536
Name: absolute_error, dtype: float64


In [39]:
monthly_diagnostic.sort_values(
    "absolute_error",
    ascending=False
).head(10)

,임대료(만원),자치구명,법정동명,임대면적,building_age,log_deposit,prediction,absolute_error
110759,80,서초구,서초동,244.500,22.0,6.216606,16047.970948,15967.970948
111034,80,서초구,서초동,244.500,22.0,6.216606,16047.970948,15967.970948
61987,1000,성동구,성수동1가,273.928,5.0,13.604791,10340.288024,9340.288024
62242,1000,성동구,성수동1가,273.928,5.0,13.604791,10340.288024,9340.288024
89143,4000,성동구,성수동1가,241.930,14.0,9.210440,12288.591411,8288.591411
89462,4000,성동구,성수동1가,241.930,14.0,9.210440,12288.591411,8288.591411
53375,1400,송파구,신천동,244.980,12.0,9.903538,9370.113069,7970.113069
53058,1400,송파구,신천동,244.980,12.0,9.903538,9370.113069,7970.113069
107308,1150,강남구,청담동,244.770,24.0,9.210440,8869.972644,7719.972644
106920,1150,강남구,청담동,244.770,24.0,9.210440,8869.972644,7719.972644


### 월세 Ridge Regression 오차 진단

Ridge Regression은 Median Baseline보다 MAE는 감소했지만,
RMSE와 R²는 오히려 크게 악화되었다.

예측값 분포를 확인한 결과:

- 실제 월세 99.9% 분위수: 1,200만원
- 예측 월세 99.9% 분위수: 약 2,725만원
- 실제 월세 최대값: 4,000만원
- 예측 월세 최대값: 약 16,048만원

일부 극단적인 주택 특성에서
Ridge Regression이 월세를 과도하게 크게 예측하는 현상이 확인되었다.

특히 넓은 전용면적과 낮은 보증금처럼
일반적인 계약과 다른 Feature 조합에서 큰 오차가 발생하였다.

선형 모델은 각 변수의 효과를 선형적으로 결합하기 때문에
이러한 특이한 변수 조합을 충분히 표현하지 못하며,
로그 Target을 원래 가격 단위로 역변환하는 과정에서
큰 예측값이 더욱 확대되는 문제가 나타날 수 있다.

따라서 월세 예측에서는 비선형 관계와 변수 간 상호작용을
학습할 수 있는 Tree 기반 모델의 성능을 추가로 확인한다.

## 월세 Random Forest

Ridge Regression의 오차 분석에서 일부 특이한 Feature 조합에 대해
과도한 예측값이 발생하는 문제가 확인되었다.

Random Forest는 변수의 관계를 선형식으로 가정하지 않고
여러 결정트리를 이용해 비선형 관계와 변수 간 상호작용을 학습한다.

따라서 동일한 Train/Test 데이터를 이용하여
Ridge Regression의 극단적인 예측 문제가 감소하는지 확인한다.

In [40]:
monthly_rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            monthly_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=5,
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

In [41]:
monthly_rf_model.fit(
    X_monthly_train,
    y_monthly_train_log
);

In [42]:
monthly_rf_pred_log = monthly_rf_model.predict(
    X_monthly_test
)

monthly_rf_pred = np.expm1(
    monthly_rf_pred_log
)

In [43]:
monthly_rf_result = evaluate_regression(
    y_monthly_test,
    monthly_rf_pred,
    "Monthly Random Forest"
)

[Monthly Random Forest]
MAE : 29.88
RMSE: 59.86
R²  : 0.7722


In [44]:
monthly_rf_vs_ridge = (
    monthly_ridge_result["MAE"]
    - monthly_rf_result["MAE"]
) / monthly_ridge_result["MAE"] * 100

monthly_rf_vs_baseline = (
    monthly_baseline_result["MAE"]
    - monthly_rf_result["MAE"]
) / monthly_baseline_result["MAE"] * 100

print(
    "Ridge 대비 MAE 개선율:",
    f"{monthly_rf_vs_ridge:.2f}%"
)

print(
    "Baseline 대비 MAE 개선율:",
    f"{monthly_rf_vs_baseline:.2f}%"
)

Ridge 대비 MAE 개선율: 48.32%
Baseline 대비 MAE 개선율: 60.24%


In [45]:
print(
    "Random Forest 예측 최대:",
    monthly_rf_pred.max()
)

Random Forest 예측 최대: 2213.2219936949336


### 월세 Random Forest 결과

Random Forest의 월세 예측 성능은 다음과 같다.

- MAE: 29.88만원
- RMSE: 59.86만원
- R²: 0.7722
- Ridge 대비 MAE 개선율: 48.32%
- Median Baseline 대비 MAE 개선율: 60.24%

Random Forest는 Ridge Regression보다
MAE와 RMSE 모두 크게 감소하였으며,
R² 역시 -0.8308에서 0.7722로 크게 개선되었다.

또한 Ridge Regression에서 약 16,048만원까지 발생했던
극단적인 예측값이 Random Forest에서는
약 2,213만원 수준으로 감소하였다.

이는 월세가격이 보증금, 면적, 지역 등의 변수와
단순한 선형 관계만을 가지는 것이 아니라,
변수 간 비선형 관계와 상호작용의 영향을 크게 받는다는 것을 보여준다.

따라서 월세 예측에서도 Tree 기반 모델이
선형 모델보다 적합한 것으로 판단된다.

## 월세 XGBoost

Random Forest에서 비선형 모델의 성능 향상이 확인되었다.

이번에는 Gradient Boosting 기반의 XGBoost를 적용하여
Random Forest보다 추가적인 성능 향상이 가능한지 확인한다.

전세 모델과 동일하게 여러 트리가 순차적으로
이전 모델의 오차를 보완하도록 학습한다.

In [46]:
monthly_xgb_model = Pipeline(
    steps=[
        (
            "preprocessor",
            monthly_preprocessor
        ),
        (
            "model",
            XGBRegressor(
                n_estimators=400,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                tree_method="hist",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

In [47]:
monthly_xgb_model.fit(
    X_monthly_train,
    y_monthly_train_log
);

In [48]:
monthly_xgb_pred_log = monthly_xgb_model.predict(
    X_monthly_test
)

monthly_xgb_pred = np.expm1(
    monthly_xgb_pred_log
)

In [50]:
monthly_xgb_result = evaluate_regression(
    y_monthly_test,
    monthly_xgb_pred,
    "Monthly XGBoost"
)

[Monthly XGBoost]
MAE : 29.32
RMSE: 57.68
R²  : 0.7885


In [51]:
monthly_xgb_vs_rf = (
    monthly_rf_result["MAE"]
    - monthly_xgb_result["MAE"]
) / monthly_rf_result["MAE"] * 100

monthly_xgb_vs_baseline = (
    monthly_baseline_result["MAE"]
    - monthly_xgb_result["MAE"]
) / monthly_baseline_result["MAE"] * 100

print(
    "Random Forest 대비 MAE 개선율:",
    f"{monthly_xgb_vs_rf:.2f}%"
)

print(
    "Baseline 대비 MAE 개선율:",
    f"{monthly_xgb_vs_baseline:.2f}%"
)

print(
    "XGBoost 예측 최대:",
    monthly_xgb_pred.max()
)

Random Forest 대비 MAE 개선율: 1.86%
Baseline 대비 MAE 개선율: 60.98%
XGBoost 예측 최대: 3095.6833


### 월세 XGBoost 결과

XGBoost의 월세 예측 성능은 다음과 같다.

- MAE: 29.32만원
- RMSE: 57.68만원
- R²: 0.7885
- Random Forest 대비 MAE 개선율: 1.86%
- Median Baseline 대비 MAE 개선율: 60.98%

현재 비교한 모델 가운데 XGBoost가
가장 낮은 MAE와 RMSE, 가장 높은 R²를 기록하였다.

Random Forest 대비 MAE 개선폭은 약 1.9%로 크지는 않았지만,
RMSE가 59.86만원에서 57.68만원으로 감소하고
R²도 0.7722에서 0.7885로 향상되어
전체적인 예측 성능은 XGBoost가 가장 우수했다.

또한 XGBoost의 최대 예측값은 약 3,096만원으로,
Ridge Regression에서 발생했던 약 16,048만원 수준의
극단적인 과대예측 현상이 크게 완화되었다.

따라서 현재 Feature와 모델 구성에서는
XGBoost를 월세가격 예측의 최종 후보 모델로 선정한다.

In [52]:
monthly_model_results = pd.DataFrame([
    monthly_baseline_result,
    monthly_ridge_result,
    monthly_rf_result,
    monthly_xgb_result
])

monthly_model_results = (
    monthly_model_results
    .sort_values("MAE")
    .reset_index(drop=True)
)

monthly_model_results

,Model,MAE,RMSE,R2
0,Monthly XGBoost,29.320686,57.677033,0.788521
1,Monthly Random Forest,29.876813,59.864113,0.772178
2,Monthly Ridge Regression,57.809443,169.703334,-0.830808
3,Monthly Median Baseline,75.149131,131.832323,-0.104857


In [53]:
final_model_summary = pd.DataFrame({
    "Target": [
        "전세보증금",
        "월세"
    ],
    "Final Model": [
        "XGBoost",
        "XGBoost"
    ],
    "MAE": [
        xgb_result["MAE"],
        monthly_xgb_result["MAE"]
    ],
    "RMSE": [
        xgb_result["RMSE"],
        monthly_xgb_result["RMSE"]
    ],
    "R2": [
        xgb_result["R2"],
        monthly_xgb_result["R2"]
    ],
    "Baseline MAE Improvement (%)": [
        (
            baseline_result["MAE"]
            - xgb_result["MAE"]
        ) / baseline_result["MAE"] * 100,

        (
            monthly_baseline_result["MAE"]
            - monthly_xgb_result["MAE"]
        ) / monthly_baseline_result["MAE"] * 100
    ]
})

final_model_summary

,Target,Final Model,MAE,RMSE,R2,Baseline MAE Improvement (%)
0,전세보증금,XGBoost,9074.272461,14690.369635,0.840278,62.838884
1,월세,XGBoost,29.320686,57.677033,0.788521,60.983333


## 모델링 종합 결론

서울 아파트 전월세 가격 예측을 위해
Median Baseline, Ridge Regression, Random Forest, XGBoost를 비교하였다.

### 전세가격 예측

최종 모델인 XGBoost는

- MAE: 9,074.27만원
- RMSE: 14,690.37만원
- R²: 0.8403

을 기록하였다.

Median Baseline 대비 MAE가 약 62.8% 감소했으며,
2025년 Test 데이터의 전세가격 변동 중 약 84.0%를 설명하였다.

### 월세가격 예측

최종 모델인 XGBoost는

- MAE: 29.32만원
- RMSE: 57.68만원
- R²: 0.7885

를 기록하였다.

Median Baseline 대비 MAE가 약 61.0% 감소했으며,
2025년 Test 데이터의 월세가격 변동 중 약 78.9%를 설명하였다.

전세와 월세 모두 선형 모델보다
Random Forest와 XGBoost 같은 Tree 기반 모델의 성능이 높게 나타났다.

이는 서울 아파트 임대가격이 면적, 지역, 건축연식,
보증금 및 시간 변수와 단순한 선형 관계만을 가지는 것이 아니라,
변수 간 비선형 관계와 상호작용의 영향을 받는다는 것을 시사한다.

최종적으로 전세가격과 월세가격 예측 모두
XGBoost를 최종 모델로 선정하였다.

다음 단계에서는 SHAP을 이용하여
최종 모델의 예측에 각 Feature가 어떤 영향을 미치는지 분석한다.

In [54]:
from pathlib import Path
import joblib

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    xgb_model,
    MODEL_DIR / "jeonse_xgboost.joblib"
)

joblib.dump(
    monthly_xgb_model,
    MODEL_DIR / "monthly_xgboost.joblib"
)

print("최종 모델 저장 완료")

최종 모델 저장 완료
